# detección shows 2026 
para mejorar la definición de artistas inactivos.
Se hace sobre Chartmetric. 
input: sin_shows_2026_checked_clean.csv
output: sin_shows_2026_checked_clean.csv
------------------
si shows 2026 > 0
show_2026 == true
has_live_data == true
[recordemos que se relevaron shows 2024 y 2025 y que NO HAY información disponible sobre shows previos en la base de datos de chartmetric confirmado por escrito por su staff]
------------------
Total artistas en archivo: 2310

Shows 2026 confirmados: 223
Sin shows detectados: 2087
Pendientes (api_status NA): 0

Estado de API:
api_status
ok    2310

In [ ]:
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

# =========================
# CONFIGURACIÓN
# =========================

#INPUT_CSV = Path(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows.csv")
INPUT_CSV = Path(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked.csv")

OUTPUT_CSV = Path(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked.csv")

HOST = "https://api.chartmetric.com"
REFRESH_TOKEN = "MI TOKEN" #original

#REFRESH_TOKEN = "MI TOKEN" #brave

SLEEP_SECONDS = 1.1
LIMIT = 100


# =========================
# FUNCIONES
# =========================

def get_access_token(refresh_token):
    res = requests.post(
        f"{HOST}/api/token",
        json={"refreshtoken": refresh_token},
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    res.raise_for_status()
    return res.json()["token"]


def build_headers(access_token):
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }


def safe_get(url, headers, params, refresh_token, max_retries=3):
    current_headers = headers.copy()

    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers=current_headers, params=params, timeout=60)

            if res.status_code == 401:
                new_token = get_access_token(refresh_token)
                current_headers = build_headers(new_token)
                res = requests.get(url, headers=current_headers, params=params, timeout=60)

            if res.status_code == 429:
                wait_time = 2 + attempt * 2
                print(f"  429 Too Many Requests. Espera {wait_time}s")
                print("  X-RateLimit-Limit:", res.headers.get("X-RateLimit-Limit"))
                print("  X-RateLimit-Remaining:", res.headers.get("X-RateLimit-Remaining"))
                print("  X-RateLimit-Reset:", res.headers.get("X-RateLimit-Reset"))
                time.sleep(wait_time)
                continue


            res.raise_for_status()
            return res, current_headers

        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise e
            wait_time = 2 + attempt * 2
            print(f"  Reintento por error: {e}. Espera {wait_time}s")
            time.sleep(wait_time)

    raise RuntimeError("No se pudo completar la request.")


def days_ago_from_today(target_date):
    return (date.today() - target_date).days


def get_2026_windows():
    today = date.today()
    jan_1 = date(2026, 1, 1)
    dec_31 = date(2026, 12, 31)

    past_end = min(today, dec_31)

    past_window = {
        "fromDaysAgo": days_ago_from_today(jan_1),
        "toDaysAgo": max(days_ago_from_today(past_end), 0),
    }

    current_window = {
        "fromDaysAgo": 0,
        "toDaysAgo": -max((dec_31 - today).days, 0),
    }

    return {
        "past": past_window,
        "current": current_window
    }


def extract_events(payload):
    obj = payload.get("obj", {})

    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):
        if "data" in obj and isinstance(obj["data"], list):
            return obj["data"]
        if "events" in obj and isinstance(obj["events"], list):
            return obj["events"]
        if "obj" in obj and isinstance(obj["obj"], list):
            return obj["obj"]

    return []


def extract_total(payload, events):
    obj = payload.get("obj", {})

    if isinstance(obj, dict):
        for key in ["total", "length", "count", "totalItems", "total_items"]:
            if key in obj and obj[key] is not None:
                try:
                    return int(obj[key])
                except Exception:
                    pass

    return len(events)


def get_event_key(event):
    for key in ["id", "event_id", "cm_event", "uuid"]:
        if key in event and event[key] is not None:
            return f"{key}:{event[key]}"

    event_name = event.get("name") or event.get("event") or ""
    event_date = event.get("start_date") or event.get("date") or event.get("datetime") or ""

    venue_name = ""
    venue = event.get("venue")
    if isinstance(venue, dict):
        venue_name = venue.get("name", "")
    else:
        venue_name = event.get("venue_name", "")

    return f"fallback:{event_name}|{event_date}|{venue_name}"


def fetch_all_events_for_status(artist_id, status, headers, refresh_token, window, limit=100):
    url = f"{HOST}/api/artist/{artist_id}/{status}/events"
    offset = 0
    all_events = []
    current_headers = headers

    while True:
        params = {
            "fromDaysAgo": window["fromDaysAgo"],
            "toDaysAgo": window["toDaysAgo"],
            "limit": limit,
            "offset": offset,
        }

        res, current_headers = safe_get(url, current_headers, params, refresh_token)
        payload = res.json()

        events = extract_events(payload)
        total = extract_total(payload, events)

        all_events.extend(events)

        if len(events) < limit:
            break

        offset += limit

        if total > 0 and offset >= total:
            break

        time.sleep(SLEEP_SECONDS)

    return all_events, current_headers


def fetch_2026_events_for_artist(artist_id, headers, refresh_token):
    windows = get_2026_windows()
    current_headers = headers
    all_events = []

    for status in ["past", "current"]:
        events, current_headers = fetch_all_events_for_status(
            artist_id=artist_id,
            status=status,
            headers=current_headers,
            refresh_token=refresh_token,
            window=windows[status],
            limit=LIMIT
        )
        all_events.extend(events)
        time.sleep(SLEEP_SECONDS)

    dedup = {}
    for event in all_events:
        dedup[get_event_key(event)] = event

    return list(dedup.values()), current_headers


# =========================
# CARGA DEL CSV
# =========================

df = pd.read_csv(INPUT_CSV)

#para continuidad
df_pending = df[df["api_status"].isna() | df["api_status"].astype(str).str.startswith("error")].copy()

print("Total artistas:", len(df))
print("Pendientes:", len(df_pending))

required_cols = [
    "chartmetric_id",
    "artist_name",
    "country",
    "band",
    "primary_genre",
    "career_stage",
    "sp_followers",
    "sp_monthly_listeners",
    "cm_artist_rank",
    "muerto_disuelto"
]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en el CSV: {missing_cols}")

df["chartmetric_id"] = pd.to_numeric(df["chartmetric_id"], errors="coerce")
df = df.dropna(subset=["chartmetric_id"]).copy()
df["chartmetric_id"] = df["chartmetric_id"].astype(int)

print("Total de artistas a consultar:", len(df_pending))


# =========================
# AUTENTICACIÓN
# =========================

access_token = get_access_token(REFRESH_TOKEN)
headers = build_headers(access_token)

time.sleep(2.0)

# =========================
# LOOP PRINCIPAL
# =========================

results = []
current_headers = headers

#for i, row in enumerate(df.itertuples(index=False), start=1):
#para continuidad
for i, row in enumerate(df_pending.itertuples(index=False), start=1):

    artist_id = int(row.chartmetric_id)
    artist_name = row.artist_name

    print(f"[{i}/{len(df_pending)}] {artist_name} ({artist_id})")

    try:
        events_2026, current_headers = fetch_2026_events_for_artist(
            artist_id=artist_id,
            headers=current_headers,
            refresh_token=REFRESH_TOKEN
        )

        n_shows_2026 = len(events_2026)

        if n_shows_2026 > 0:
            results.append({
                "chartmetric_id": artist_id,
                "artist_name": artist_name,
                "n_shows_2026": n_shows_2026,
                "shows_2026": True,
                "has_live_data": True,
                "api_status": "ok"
            })
            print(f"  n_shows_2026={n_shows_2026}")

        else:
            results.append({
                "chartmetric_id": artist_id,
                "artist_name": artist_name,
                "n_shows_2026": pd.NA,
                "shows_2026": pd.NA,
                "has_live_data": False,
                "api_status": "ok"
            })
            print("  sin shows en 2026")

    except Exception as e:
        results.append({
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "n_shows_2026": pd.NA,
            "shows_2026": pd.NA,
            "has_live_data": pd.NA,
            "api_status": f"error: {str(e)}"
        })

        print(f"  ERROR: {e}")

    time.sleep(SLEEP_SECONDS)

    if i % 25 == 0:
        df_partial = pd.DataFrame(results)
        df_out_partial = df.merge(df_partial, on=["chartmetric_id", "artist_name"], how="left")
        df_out_partial.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"  Guardado parcial en: {OUTPUT_CSV}")


# =========================
# GUARDADO FINAL
# =========================

df_results = pd.DataFrame(results)

expected_cols = [
    "chartmetric_id",
    "artist_name",
    "n_shows_2026",
    "shows_2026",
    "has_live_data",
    "api_status"
]

for col in expected_cols:
    if col not in df_results.columns:
        df_results[col] = pd.NA

df_out = df.merge(
    df_results[expected_cols],
    on=["chartmetric_id", "artist_name"],
    how="left"
)

df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nProceso terminado.")
print("Archivo guardado en:", OUTPUT_CSV)

print("\nResumen api_status:")
print(df_results["api_status"].value_counts(dropna=False))

ok_mask = df_results["api_status"] == "ok"
true_mask = df_results["shows_2026"] == True
false_live_mask = (df_results["api_status"] == "ok") & (df_results["has_live_data"] == False)

print("\nArtistas con shows en 2026:", int(true_mask.sum()))
print("Artistas sin shows en 2026:", int(false_live_mask.sum()))
print("Artistas con error de API:", int((df_results["api_status"] != "ok").sum()))

Total artistas: 2310
Pendientes: 1810
Total de artistas a consultar: 2310
[1/1810] Bsmalla (9382978)
  sin shows en 2026
[2/1810] Lisa Stansfield (177015)
  sin shows en 2026
[3/1810] John Roa (161604)
  sin shows en 2026
[4/1810] Samar Singh (3480460)
  sin shows en 2026
[5/1810] Madame (1371988)
  n_shows_2026=3
[6/1810] Matt Redman (949)
  sin shows en 2026
[7/1810] Umair (4261721)
  sin shows en 2026
[8/1810] Barasuara (415540)
  sin shows en 2026
[9/1810] Oshima (8193644)
  sin shows en 2026
[10/1810] Joachim Garraud (308957)
  n_shows_2026=1
[11/1810] thiarajxtt (9036630)
  sin shows en 2026
[12/1810] Tribalistas (208242)
  sin shows en 2026
[13/1810] Carolina Gaitán - La Gaita (644506)
  sin shows en 2026
[14/1810] Laberinto (493766)
  sin shows en 2026
[15/1810] Gavin Magnus (1494677)
  n_shows_2026=1
[16/1810] Kerispatih (590638)
  sin shows en 2026
[17/1810] Ada Band (182524)
  sin shows en 2026
[18/1810] Rick & Renner (57687)
  sin shows en 2026
[19/1810] SOOBIN (495072)
  s

KeyboardInterrupt: 

# LIMPIAR CSV 1 vez
porque se habían duplicado columnas con la primera versio ndel código

In [3]:
import pandas as pd
from pathlib import Path

INPUT_CSV = Path(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked.csv")
OUTPUT_CLEAN = Path(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked_clean.csv")

def clean_duplicate_output_columns(df):
    target_cols = ["n_shows_2026", "shows_2026", "has_live_data", "api_status"]

    for col in target_cols:
        candidates = [c for c in df.columns if c == col or c.startswith(col + "_")]
        if not candidates:
            df[col] = pd.NA
            continue

        if col not in df.columns:
            df[col] = pd.NA

        preferred_order = [c for c in candidates if c.endswith("_y")] + \
                          [c for c in candidates if c == col] + \
                          [c for c in candidates if c.endswith("_x")] + \
                          [c for c in candidates if c not in [col]]

        for candidate in preferred_order:
            if candidate == col:
                continue
            df[col] = df[col].combine_first(df[candidate])

        cols_to_drop = [c for c in candidates if c != col]
        if cols_to_drop:
            df = df.drop(columns=cols_to_drop)

    return df

df = pd.read_csv(INPUT_CSV)
df = clean_duplicate_output_columns(df)

df.to_csv(OUTPUT_CLEAN, index=False, encoding="utf-8-sig")

print("Archivo limpio guardado en:")
print(OUTPUT_CLEAN)
print("\nColumnas finales:")
print(df.columns.tolist())

Archivo limpio guardado en:
C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked_clean.csv

Columnas finales:
['chartmetric_id', 'artist_name', 'country', 'band', 'primary_genre', 'career_stage', 'sp_followers', 'sp_monthly_listeners', 'cm_artist_rank', 'muerto_disuelto', 'n_shows_2026', 'shows_2026', 'has_live_data', 'api_status']


In [4]:
df = pd.read_csv("sin_shows_2026_checked_clean.csv")

print(df.columns)
print(df["api_status"].value_counts(dropna=False))

Index(['chartmetric_id', 'artist_name', 'country', 'band', 'primary_genre',
       'career_stage', 'sp_followers', 'sp_monthly_listeners',
       'cm_artist_rank', 'muerto_disuelto', 'n_shows_2026', 'shows_2026',
       'has_live_data', 'api_status'],
      dtype='str')
api_status
NaN                                        1260
ok                                         1000
error: No se pudo completar la request.      50
Name: count, dtype: int64


# borrar SIN SHOWS 2026 CHECKED

# NUEVO CODIGO PARA CONTINUAR
SIN DUPLICAR COLUMNAS

In [ ]:
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

# =========================
# CONFIGURACIÓN
# =========================

INPUT_CSV = Path(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked_clean.csv")
OUTPUT_CSV = INPUT_CSV

HOST = "https://api.chartmetric.com"
REFRESH_TOKEN = "mi token"

SLEEP_SECONDS = 1.1
LIMIT = 100


# =========================
# FUNCIONES
# =========================

def get_access_token(refresh_token):
    res = requests.post(
        f"{HOST}/api/token",
        json={"refreshtoken": refresh_token},
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    res.raise_for_status()
    return res.json()["token"]


def build_headers(access_token):
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }


def safe_get(url, headers, params, refresh_token, max_retries=3):

    current_headers = headers.copy()

    for attempt in range(max_retries):

        try:

            res = requests.get(url, headers=current_headers, params=params, timeout=60)

            if res.status_code == 401:

                new_token = get_access_token(refresh_token)
                current_headers = build_headers(new_token)

                res = requests.get(url, headers=current_headers, params=params, timeout=60)

            if res.status_code == 429:

                wait_time = 2 + attempt * 2

                print(f"  429 Too Many Requests. Espera {wait_time}s")
                print("  X-RateLimit-Limit:", res.headers.get("X-RateLimit-Limit"))
                print("  X-RateLimit-Remaining:", res.headers.get("X-RateLimit-Remaining"))
                print("  X-RateLimit-Reset:", res.headers.get("X-RateLimit-Reset"))

                time.sleep(wait_time)

                continue

            res.raise_for_status()

            return res, current_headers

        except requests.RequestException as e:

            if attempt == max_retries - 1:
                raise e

            wait_time = 2 + attempt * 2

            print(f"  Reintento por error: {e}. Espera {wait_time}s")

            time.sleep(wait_time)

    raise RuntimeError("No se pudo completar la request.")


def days_ago_from_today(target_date):
    return (date.today() - target_date).days


def get_2026_windows():

    today = date.today()

    jan_1 = date(2026, 1, 1)
    dec_31 = date(2026, 12, 31)

    past_end = min(today, dec_31)

    past_window = {
        "fromDaysAgo": days_ago_from_today(jan_1),
        "toDaysAgo": max(days_ago_from_today(past_end), 0),
    }

    current_window = {
        "fromDaysAgo": 0,
        "toDaysAgo": -max((dec_31 - today).days, 0),
    }

    return {
        "past": past_window,
        "current": current_window
    }


def extract_events(payload):

    obj = payload.get("obj", {})

    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):

        if "data" in obj and isinstance(obj["data"], list):
            return obj["data"]

        if "events" in obj and isinstance(obj["events"], list):
            return obj["events"]

        if "obj" in obj and isinstance(obj["obj"], list):
            return obj["obj"]

    return []


def get_event_key(event):

    for key in ["id", "event_id", "cm_event", "uuid"]:

        if key in event and event[key] is not None:
            return f"{key}:{event[key]}"

    event_name = event.get("name") or event.get("event") or ""
    event_date = event.get("start_date") or event.get("date") or event.get("datetime") or ""

    return f"fallback:{event_name}|{event_date}"


def fetch_all_events_for_status(artist_id, status, headers, refresh_token, window, limit=100):

    url = f"{HOST}/api/artist/{artist_id}/{status}/events"

    offset = 0
    all_events = []
    current_headers = headers

    while True:

        params = {
            "fromDaysAgo": window["fromDaysAgo"],
            "toDaysAgo": window["toDaysAgo"],
            "limit": limit,
            "offset": offset,
        }

        res, current_headers = safe_get(url, current_headers, params, refresh_token)

        payload = res.json()

        events = extract_events(payload)

        all_events.extend(events)

        if len(events) < limit:
            break

        offset += limit

        time.sleep(SLEEP_SECONDS)

    return all_events, current_headers


def fetch_2026_events_for_artist(artist_id, headers, refresh_token):

    windows = get_2026_windows()

    current_headers = headers
    all_events = []

    for status in ["past", "current"]:

        events, current_headers = fetch_all_events_for_status(
            artist_id=artist_id,
            status=status,
            headers=current_headers,
            refresh_token=refresh_token,
            window=windows[status],
            limit=LIMIT
        )

        all_events.extend(events)

        time.sleep(SLEEP_SECONDS)

    dedup = {}

    for event in all_events:
        dedup[get_event_key(event)] = event

    return list(dedup.values()), current_headers


# =========================
# CARGA DEL CSV
# =========================

df = pd.read_csv(INPUT_CSV)

for col in ["n_shows_2026", "shows_2026", "has_live_data", "api_status"]:

    if col not in df.columns:
        df[col] = pd.NA


df_pending = df[df["api_status"].isna() | df["api_status"].astype(str).str.startswith("error")].copy()

print("Total artistas:", len(df))
print("Pendientes:", len(df_pending))


# =========================
# AUTENTICACIÓN
# =========================

access_token = get_access_token(REFRESH_TOKEN)

headers = build_headers(access_token)

time.sleep(2)


# =========================
# LOOP PRINCIPAL
# =========================

current_headers = headers

for i, row in enumerate(df_pending.itertuples(index=False), start=1):

    artist_id = int(row.chartmetric_id)
    artist_name = row.artist_name

    print(f"[{i}/{len(df_pending)}] {artist_name} ({artist_id})")

    row_mask = (df["chartmetric_id"] == artist_id) & (df["artist_name"] == artist_name)

    try:

        events_2026, current_headers = fetch_2026_events_for_artist(
            artist_id=artist_id,
            headers=current_headers,
            refresh_token=REFRESH_TOKEN
        )

        n_shows_2026 = len(events_2026)

        if n_shows_2026 > 0:

            df.loc[row_mask, "n_shows_2026"] = n_shows_2026
            df.loc[row_mask, "shows_2026"] = True
            df.loc[row_mask, "has_live_data"] = True
            df.loc[row_mask, "api_status"] = "ok"

            print(f"  n_shows_2026={n_shows_2026}")

        else:

            df.loc[row_mask, "n_shows_2026"] = pd.NA
            df.loc[row_mask, "shows_2026"] = pd.NA
            df.loc[row_mask, "has_live_data"] = False
            df.loc[row_mask, "api_status"] = "ok"

            print("  sin shows en 2026")

    except Exception as e:

        df.loc[row_mask, "n_shows_2026"] = pd.NA
        df.loc[row_mask, "shows_2026"] = pd.NA
        df.loc[row_mask, "has_live_data"] = pd.NA
        df.loc[row_mask, "api_status"] = f"error: {str(e)}"

        print(f"  ERROR: {e}")

    time.sleep(SLEEP_SECONDS)

    if i % 25 == 0:

        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        print(f"  Guardado parcial en: {OUTPUT_CSV}")


# =========================
# GUARDADO FINAL
# =========================

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nProceso terminado.")
print("Archivo guardado en:", OUTPUT_CSV)

print("\nResumen api_status:")

print(df["api_status"].value_counts(dropna=False))

print("\nArtistas con shows 2026:", (df["shows_2026"] == True).sum())
print("Artistas sin shows 2026:", (df["has_live_data"] == False).sum())

Total artistas: 2310
Pendientes: 320
[1/320] HUNTR/X (13915334)
  sin shows en 2026
[2/320] Pritam (127583)
  sin shows en 2026
[3/320] Pharrell Williams (2143)
  sin shows en 2026
[4/320] ROSÉ (4114519)
  sin shows en 2026
[5/320] Ariana Grande (3963)
  n_shows_2026=41
[6/320] Jung Kook (8467662)
  sin shows en 2026
[7/320] Arctic Monkeys (2289)
  sin shows en 2026
[8/320] Bizarrap (648982)
  sin shows en 2026
[9/320] Tunchee Stan (4395173)
  sin shows en 2026
[10/320] A$AP Rocky (4015)
  n_shows_2026=42
[11/320] Disney (303089)
  n_shows_2026=20
[12/320] Frank Ocean (3849)
  sin shows en 2026
[13/320] Dr. Dre (129)
  sin shows en 2026
[14/320] Indila (21326)
  sin shows en 2026
[15/320] Agust D (1404760)
  sin shows en 2026
[16/320] Tulsi Kumar (35325)
  sin shows en 2026
[17/320] 6ix9ine (749249)
  sin shows en 2026
[18/320] MC Meno K (4164247)
  sin shows en 2026
[19/320] Dido (407)
  sin shows en 2026
[20/320] Gnarls Barkley (2579)
  sin shows en 2026
[21/320] Macklemore & Ryan Le

# REVISION HALLAZGOS

In [21]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked_clean.csv")

print("Total artistas en archivo:", len(df))
print()

print("Shows 2026 confirmados:", (df["shows_2026"] == True).sum())
print("Sin shows detectados:", (df["has_live_data"] == False).sum())
print("Pendientes (api_status NA):", df["api_status"].isna().sum())
print()

print("Estado de API:")
print(df["api_status"].value_counts(dropna=False))

Total artistas en archivo: 2310

Shows 2026 confirmados: 223
Sin shows detectados: 2087
Pendientes (api_status NA): 0

Estado de API:
api_status
ok    2310
Name: count, dtype: int64


In [22]:
df[df["shows_2026"] == True][
    ["artist_name","chartmetric_id","n_shows_2026","cm_artist_rank"]
].sort_values("n_shows_2026", ascending=False).head(20)

,artist_name,chartmetric_id,n_shows_2026,cm_artist_rank
576,Florent Pagny,48642,118.0,8555
1376,Orelsan,309041,92.0,4225
151,Black Label Society,68505,74.0,3083
1973,Miguel,3853,72.0,310
523,Alter Bridge,2051,70.0,6203
1982,Harry Styles,558681,69.0,23
437,Tori Amos,89877,66.0,9653
39,The Neighbourhood,4298,65.0,237
1852,Lily Allen,2615,63.0,1421
1936,Hilary Duff,1614,58.0,1805


In [12]:
print(
    "Porcentaje con shows 2026:",
    round((df["shows_2026"] == True).mean()*100,2),
    "%"
)

Porcentaje con shows 2026: 3.68 %
